In [3]:
import datetime
from monthdelta import monthmod
from dateutil.relativedelta import relativedelta
import requests
from bs4 import BeautifulSoup
import time
import random
import my_snipets
import pandas as pd
import os
import re
from io import StringIO

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions
from selenium.webdriver.chrome.options import Options

from tqdm.notebook import tqdm

In [207]:
class makeRaceData:

    def __init__(self, startYM:str, endYM:str):
        self.startYM = startYM
        self.endYM = endYM
        print(f"{self.startYM}～{self.endYM}のレース情報のデータを作成します。")

    def makeYMList(self):
        """
        (year,month)のリストを作成するメソッド
        """
        self.YMList = []

        start = datetime.date(int(self.startYM[0:4]),int(self.startYM[4:]),1)
        end = datetime.date(int(self.endYM[0:4]),int(self.endYM[4:]),1)
        diff = monthmod(start,end)

        for i in tqdm(range(0,diff[0].months + 1),desc="makeYMList"):
            tgt = start + relativedelta(months=i)
            tgtYM = (str(tgt.year),str(tgt.month).zfill(2))
            self.YMList.append(tgtYM)


    def makeRaceDateURLList(self):
        """
        対象日付の開催レース一覧ページのURLを生成する関数
        `date`: 対象の年月日 (datetime.dateオブジェクト)
        """
        self.raceDateURLList = []
        for year , month in tqdm(self.YMList,desc="makeRaceDateURLList"):
            #対象年月の開催カレンダーのURLを生成
            url = f"https://race.netkeiba.com/top/calendar.html?year={year}&month={month}"
            #カレンダーページのHTMLを取得
            headers = {'User-Agent': random.choice(my_snipets.makeUserAgents())}
            html = requests.get(url, headers=headers)
            html.encoding = "EUC-JP"

            #HTMLをsoupオブジェクトに変換
            soup = BeautifulSoup(html.text)

            #開催日の一覧を取得し、開催日ページのURLを取得
            RaceCellBoxAll = soup.find_all("td",attrs={"class":"RaceCellBox"})
            
            for i in range(len(RaceCellBoxAll)):
                if RaceCellBoxAll[i].find("a") is not None:
                    self.raceDateURLList.append(RaceCellBoxAll[i].find("a")["href"].replace("..","https://race.netkeiba.com/"))

            time.sleep(2)

    def makeRaceURLList(self):
        """
        開催日ページのURLから、レースIDのリストを作成する関数
        """

        self.raceIDList = []

        with tqdm(self.raceDateURLList) as urlPbar:
            urlPbar.set_description(desc="makeRaceURLList")
            for tgtURL in urlPbar:
                dateStr = tgtURL.split("=")[1]
                #urlPbar.set_postfix(dateStr)
                # Seleniumを使用してブラウザを起動し、対象URLにアクセス
                option = Options()
                option.add_argument('--headless')
                browser = webdriver.Chrome(options=option)
                browser.get(tgtURL)

                # ブラウザから必要な情報を取得
                RaceListDataItemAll = browser.find_elements(By.CLASS_NAME,"RaceList_DataItem")
                for item in tqdm(RaceListDataItemAll,desc=dateStr, leave=False):
                    raceURL = item.find_element(By.TAG_NAME,"a").get_attribute("href")
                    self.raceIDList.append(re.sub(r"\D+", "", str(raceURL.split("/")[-1:])))

                # ブラウザを閉じる
                browser.quit()

                time.sleep(2)

    def screpingRaceHTML(self,skip:bool=True):
        """
        receIDリストに含まれるレースの結果ページのHTMLをスクレイピングする
        """
        with tqdm(self.raceIDList) as urlPbar:
            urlPbar.set_description("screpingRaceHTML")
            for raceID in urlPbar:
                urlPbar.set_postfix({"raceID":raceID})
                if skip == False or os.path.isfile(f"../data/html/race/{raceID}.txt") == False:
                    url = "https://db.netkeiba.com/race/" + raceID
                    headers = {'User-Agent': random.choice(my_snipets.makeUserAgents())}
                    html = requests.get(url, headers=headers)
                    html.encoding = "EUC-JP"

                    with open(f"../data/html/race/{raceID}.txt","w") as f:
                        f.write(html.text)

                    time.sleep(2)

    def makeRawDataRace(self,skip:bool = True):
        """
        html/raceに格納されているhtmlからレース結果/レース情報/払い戻し情報のrawデータを作成する関数
        """
        with tqdm(self.raceIDList) as urlPbar:
            urlPbar.set_description("makeRawDataRace")               
            for raceID in urlPbar:
                if skip == False or os.path.isfile(f"../data/rawData/raceResults/{raceID}.pickle") == False or os.path.isfile(f"../data/rawData/raceInfos/{raceID}.pickle") == False or os.path.isfile(f"../data/rawData/return/{raceID}.pickle") == False:
                    urlPbar.set_postfix({"raceID":raceID})
                    #共通処理======================================================================================================================
                    with open(f"../data/html/race/{raceID}.txt","r") as f:
                        html = f.read()

                    soup = BeautifulSoup(html) #htmlをsoupオブジェクトへ
                    
                    #receResultsに関する処理=======================================================================================================
                    if skip == False or os.path.isfile(f"../data/rawData/raceResults/{raceID}.pickle") == False:
                        raceResults = pd.DataFrame() #最終出力データセットの定義
                        dfRace = pd.read_html(StringIO(str(soup("table")[0])))[0] #HTMLからレース結果を取得し、DataFrameへ

                        dfRace["raceID"] = raceID

                        horseIDList = []
                        horseAList = soup.find("table",attrs={"summary":"レース結果"}).find_all("a",attrs={"href":re.compile("^/horse")})
                        for a in horseAList:
                            horseID = re.findall("\d+",a["href"])
                            horseIDList.append(horseID[0])
                            
                        dfRace["horseID"] = horseIDList

                        jockeyIDList = []
                        jockeyAList = soup.find("table",attrs={"summary":"レース結果"}).find_all("a",attrs={"href":re.compile("^/jockey")})
                        for a in jockeyAList:
                            jockeyID = re.findall("\d+",a["href"])
                            jockeyIDList.append(jockeyID[0])

                        dfRace["jockeyID"] = jockeyIDList

                        raceResults = pd.concat([raceResults,dfRace])

                        raceResults.to_pickle(f"../data/rawData/raceResults/{raceID}.pickle")

                    #receInfosに関する処理==========================================================================================================
                    if skip == False or os.path.isfile(f"../data/rawData/raceInfos/{raceID}.pickle") == False:
                        raceInfos = pd.DataFrame() #最終出力データセットの定義
                        text = soup.find("div",attrs={"class" : "data_intro"}).find_all("p")[0].text + soup.find("div",attrs={"class" : "data_intro"}).find_all("p")[1].text
                        info = re.findall("\w+",text)

                        infoDict = {}
                        infoDict["raceID"] = raceID
                        infoDict["raceName"] = soup.find("div",attrs={"class" : "data_intro"}).find_all("h1")[0].text
                        for text in info:
                            if text in ["芝","ダート"]:
                                infoDict["raceType"] = text
                            if "障" in text:
                                infoDict["raceType"] = "障害"
                            if "m" in text:
                                infoDict["course_len"] = re.findall("\d+",text)[0]
                            if text in ["良","稍重","重","不良"]:
                                infoDict["groundState"] = text
                            if text in ["曇","晴","雨","小雨","小雪","雪"]:
                                infoDict["weather"] = text
                            if "年" in text :
                                infoDict["date"] = text

                        raceInfos = pd.DataFrame(infoDict,index=[0])
                        #raceInfos = pd.concat([raceInfos, dfInfo], ignore_index=True)

                        raceInfos.to_pickle(f"../data/rawData/raceInfos/{raceID}.pickle")
                    
                    #returnTableに関する処理========================================================================================================
                    #同着の時にイレギュラーパターンがあるからそれを処理する機能の追加必須
                    if skip == False or os.path.isfile(f"../data/rawData/return/{raceID}.pickle") == False:
                        table1 = pd.read_html(StringIO(str(soup("table")[1]).replace("<br/>","or")))[0]
                        table2 = pd.read_html(StringIO(str(soup("table")[2]).replace("<br/>","or")))[0]
                        returnTable = pd.concat([table1,table2])
                        returnTable.reset_index(drop=True,inplace=True)
                        
                        betList = returnTable[0].unique()
                        for betting in betList:
                        #for rowNo in range(len(returnTable)):
                            tgt = returnTable[returnTable[0] == betting]
                            if "or" in tgt[1].to_string(index=False):        
                                for i in [1,2,3]:
                                    tgtstring = tgt[i].to_string(index=False)
                                    tgtlist = tgtstring.split("or")
                                    tgtdf = pd.DataFrame(tgtlist)
                                    tgtdf.rename(columns={0:i},inplace=True)
                                    if i == 1:
                                        returnFin = tgtdf
                                    else:
                                        returnFin = pd.merge(returnFin,tgtdf,left_index=True, right_index=True,how="left")

                                    returnFin[0] = betting

                                    returnTable = returnTable[returnTable[0] != betting]
                                    returnTable = pd.concat([returnTable,returnFin])

                        returnTable.reset_index(drop=True,inplace=True)
                        returnTable.rename(columns={0:"betting",1:"results",2:"payout",3:"popularity"},inplace=True)

                        returnTable["payout"] = returnTable["payout"].str.replace(",","").astype(int)
                        returnTable["payoutRate"] = returnTable["payout"] / 100
                        returnTable["raceID"] = raceID


                        returnTable.to_pickle(f"../data/rawData/return/{raceID}.pickle")


202501～202502のレース情報のデータを作成します。


makeYMList:   0%|          | 0/2 [00:00<?, ?it/s]

makeRaceDateURLList:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

20250105:   0%|          | 0/24 [00:00<?, ?it/s]

...

20250210:   0%|          | 0/12 [00:00<?, ?it/s]

20250215:   0%|          | 0/36 [00:00<?, ?it/s]

20250216:   0%|          | 0/36 [00:00<?, ?it/s]

20250222:   0%|          | 0/36 [00:00<?, ?it/s]

20250223:   0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/528 [00:00<?, ?it/s]

  0%|          | 0/528 [00:00<?, ?it/s]

In [ ]:
def makeRawDataRace(raceIDList,skip:bool = True):
    """
    html/raceに格納されているhtmlからレース結果/レース情報/払い戻し情報のrawデータを作成する関数
    """
    cnt = 1
    for raceID in raceIDList:
        if skip == False or os.path.isfile(f"../data/rawData/raceResults/{raceID}.pickle") == False or os.path.isfile(f"../data/rawData/raceInfos/{raceID}.pickle") == False or os.path.isfile(f"../data/rawData/return/{raceID}.pickle") == False:
            print(raceID)
            #共通処理======================================================================================================================
            with open(f"../data/html/race/{raceID}.txt","r") as f:
                html = f.read()

            soup = BeautifulSoup(html) #htmlをsoupオブジェクトへ
            
            #receResultsに関する処理=======================================================================================================
            if skip == False or os.path.isfile(f"../data/rawData/raceResults/{raceID}.pickle") == False:
                raceResults = pd.DataFrame() #最終出力データセットの定義
                try:
                    dfRace = pd.read_html(StringIO(str(soup("table")[0])))[0]
                except IndexError :
                    print(f"raceID:{raceID}の処理をスキップします。")
                    continue
                #dfRace = pd.read_html(StringIO(html))[0] #HTMLからレース結果を取得し、DataFrameへ

                dfRace["raceID"] = raceID

                horseIDList = []
                horseAList = soup.find("table",attrs={"summary":"レース結果"}).find_all("a",attrs={"href":re.compile("^/horse")})
                for a in horseAList:
                    horseID = re.findall("\d+",a["href"])
                    horseIDList.append(horseID[0])
                    
                dfRace["horseID"] = horseIDList

                jockeyIDList = []
                jockeyAList = soup.find("table",attrs={"summary":"レース結果"}).find_all("a",attrs={"href":re.compile("^/jockey")})
                for a in jockeyAList:
                    jockeyID = re.findall("\d+",a["href"])
                    jockeyIDList.append(jockeyID[0])

                dfRace["jockeyID"] = jockeyIDList

                raceResults = pd.concat([raceResults,dfRace])

                raceResults.to_pickle(f"../data/rawData/raceResults/{raceID}.pickle")

            #receInfosに関する処理==========================================================================================================
            if skip == False or os.path.isfile(f"../data/rawData/raceInfos/{raceID}.pickle") == False:
                raceInfos = pd.DataFrame() #最終出力データセットの定義
                text = soup.find("div",attrs={"class" : "data_intro"}).find_all("p")[0].text + soup.find("div",attrs={"class" : "data_intro"}).find_all("p")[1].text
                info = re.findall("\w+",text)

                infoDict = {}
                infoDict["raceID"] = raceID
                infoDict["raceName"] = soup.find("div",attrs={"class" : "data_intro"}).find_all("h1")[0].text
                for text in info:
                    if text in ["芝","ダート"]:
                        infoDict["raceType"] = text
                    if "障" in text:
                        infoDict["raceType"] = "障害"
                    if "m" in text:
                        infoDict["course_len"] = re.findall("\d+",text)[0]
                    if text in ["良","稍重","重","不良"]:
                        infoDict["groundState"] = text
                    if text in ["曇","晴","雨","小雨","小雪","雪"]:
                        infoDict["weather"] = text
                    if "年" in text :
                        infoDict["date"] = text

                raceInfos = pd.DataFrame(infoDict,index=[0])
                #raceInfos = pd.concat([raceInfos, dfInfo], ignore_index=True)

                raceInfos.to_pickle(f"../data/rawData/raceInfos/{raceID}.pickle")
            
            #returnTableに関する処理========================================================================================================
            #同着の時にイレギュラーパターンがあるからそれを処理する機能の追加必須
            if skip == False or os.path.isfile(f"../data/rawData/return/{raceID}.pickle") == False:
                #html2 = html.replace("<br />","or")
                #tables = pd.read_html(StringIO(html2))
                table1 = pd.read_html(str(soup("table")[1]).replace("<br/>","or"))[0]
                table2 = pd.read_html(str(soup("table")[2]).replace("<br/>","or"))[0]

                returnTable = pd.concat([table1,table2])
                returnTable.reset_index(drop=True,inplace=True)
                
                betList = returnTable[0].unique()
                for betting in betList:
                #for rowNo in range(len(returnTable)):
                    tgt = returnTable[returnTable[0] == betting]
                    if "or" in tgt[1].to_string(index=False):        
                        for i in [1,2,3]:
                            tgtstring = tgt[i].to_string(index=False)
                            tgtlist = tgtstring.split("or")
                            tgtdf = pd.DataFrame(tgtlist)
                            tgtdf.rename(columns={0:i},inplace=True)
                            if i == 1:
                                returnFin = tgtdf
                            else:
                                returnFin = pd.merge(returnFin,tgtdf,left_index=True, right_index=True,how="left")

                            returnFin[0] = betting

                            returnTable = returnTable[returnTable[0] != betting]
                            returnTable = pd.concat([returnTable,returnFin])

                returnTable.reset_index(drop=True,inplace=True)
                returnTable.rename(columns={0:"betting",1:"results",2:"payout",3:"popularity"},inplace=True)

                returnTable["payout"] = returnTable["payout"].str.replace(",","").astype(int)
                returnTable["payoutRate"] = returnTable["payout"] / 100
                returnTable["raceID"] = raceID


                returnTable.to_pickle(f"../data/rawData/return/{raceID}.pickle")

        cnt+=1      
    return

In [1]:
raceID="201201010506"

In [4]:
with open(f"../data/html/race/{raceID}.txt","r",encoding="shift-jis") as f:
    html = f.read()

soup = BeautifulSoup(html)

In [5]:
table1 = pd.read_html(StringIO(str(soup("table")[1]).replace("<br/>","or")))[0]
table2 = pd.read_html(StringIO(str(soup("table")[2]).replace("<br/>","or")))[0]
returnTable = pd.concat([table1,table2])
returnTable.reset_index(drop=True,inplace=True)

In [6]:
returnTable

,0,1,2,3
0,単勝,9,3740,10
1,複勝,9or3or16,680or160or510,10or1or8
2,枠連,2 - 5,800,3
3,馬連,3 - 9,5510,18
4,ワイド,3 - 9or9 - 16or3 - 16,"1,670or6,250or1,040",18or59or10
5,馬単,9 → 3,12100,41
6,三連複,3 - 9 - 16,23210,79
7,三連単,9 → 3 → 16,173940,505


In [7]:
betList = returnTable[0].unique()
for betting in betList:
    tgt = returnTable[returnTable[0] == betting]
    if "or" in tgt[1].to_string(index=False):        
        for i in [1,2,3]:
            tgtstring = tgt[i].to_string(index=False)
            tgtlist = tgtstring.split("or")
            tgtdf = pd.DataFrame(tgtlist)
            tgtdf.rename(columns={0:i},inplace=True)
            if i == 1:
                returnFin = tgtdf
            else:
                returnFin = pd.merge(returnFin,tgtdf,left_index=True, right_index=True,how="left")

            returnFin[0] = betting

            returnTable = returnTable[returnTable[0] != betting]
            returnTable = pd.concat([returnTable,returnFin])

In [8]:
returnTable

,0,1,2,3
0,単勝,9,3740,10
2,枠連,2 - 5,800,3
3,馬連,3 - 9,5510,18
5,馬単,9 → 3,12100,41
6,三連複,3 - 9 - 16,23210,79
7,三連単,9 → 3 → 16,173940,505
0,複勝,9,680,10
1,複勝,3,160,1
2,複勝,16,510,8
0,ワイド,3 - 9,"1,670",18


In [9]:
returnTable.reset_index(drop=True,inplace=True)
returnTable.rename(columns={0:"betting",1:"results",2:"payout",3:"popularity"},inplace=True)

In [16]:
returnTable["payout"] = returnTable["payout"].apply(lambda x: int(str(x).replace(",", "")))


In [19]:
returnTable

,betting,results,payout,popularity,payoutRate
0,単勝,9,3740,10,37.4
1,枠連,2 - 5,800,3,8.0
2,馬連,3 - 9,5510,18,55.1
3,馬単,9 → 3,12100,41,121.0
4,三連複,3 - 9 - 16,23210,79,232.1
5,三連単,9 → 3 → 16,173940,505,1739.4
6,複勝,9,680,10,6.8
7,複勝,3,160,1,1.6
8,複勝,16,510,8,5.1
9,ワイド,3 - 9,1670,18,16.7


In [18]:
returnTable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   betting     12 non-null     object 
 1   results     12 non-null     object 
 2   payout      12 non-null     int64  
 3   popularity  12 non-null     object 
 4   payoutRate  12 non-null     float64
dtypes: float64(1), int64(1), object(3)
memory usage: 608.0+ bytes


In [17]:
returnTable["payoutRate"] = returnTable["payout"] / 100

In [ ]:
returnTable["raceID"] = raceID

In [3]:
forHRD = makeRaceData("202501","202501")
forHRD.makeYMList()
forHRD.makeRaceDateURLList()
forHRD.makeRaceURLList()

2025/01～2025/01のレース情報のデータを作成します。


makeYMList:   0%|          | 0/1 [00:00<?, ?it/s]

makeRaceURLList: 100%|██████████| 9/9 [01:08<00:00,  7.66s/it, date=2025/01/26]


In [2]:
files = glob.glob("../data/html/ped/*")
print(f"対象頭数:{len(files)}")

対象頭数:40826


In [4]:
def makehorseIDList(files):
    """
    に指定されたレース結果からユニークなhorseIDのリストを作成する
    """
    horseIDList = []

    cnt = 1
    for file in files:
        print("\r" + f"horseIDListの作成中({cnt}/{len(files)})",end="")
        filename = str(file.split("/")[-1:])
        raceID = re.sub(r"\D+", "", filename)

        horseIDTmp = pd.read_pickle(f"../data/rawData/raceResults/{raceID}.pickle")
        horseIDTmp2 = horseIDTmp["horseID"].unique()
        horseIDList.extend(horseIDTmp2)
        cnt+=1
    
    horseIDListDF = pd.DataFrame(horseIDList)
    horseIDListFin = horseIDListDF[0].unique()

    print("\n" + f"{len(horseIDListFin)}頭分のhorseIDListを作成しました")
    return horseIDListFin

In [7]:
horseIDListPrep = []
for raceID in forHRD.raceIDList:
    raceResultsDF = pd.read_pickle(f"../data/rawData/raceResults/{raceID}.pickle")
    horseIDTmp = raceResultsDF["horseID"].unique()
    horseIDListPrep.extend(horseIDTmp)

horseIDList = pd.DataFrame(horseIDListPrep)[0].unique().tolist()

In [8]:
horseIDList[0:3]

['2022105247', '2022106581', '2022107050']

In [ ]:
horseIDList = pd.DataFrame(horseIDListPrep)[0].unique().tolist()

In [ ]:
from makeRawData import makeRaceData,makeHorseData
import pandas as pd


In [2]:
horseIDList = ['2022105247', '2022106581', '2022107050']

In [3]:
MHD = makeHorseData(horseIDList)
MHD.getHorseHTML(skipResult=False, skipPED=False)
MHD.makeHorseRawData(skipResult=False, skipPED=False)

makeHorseRawData: 100%|██████████| 3/3 [00:01<00:00,  2.37it/s, horseID=2022107050(血統情報)]


#馬の

In [3]:
files = glob.glob("../data/rawData/raceResults/*")
print(f"{len(files)}レース分の結果ファイルのリストを作成しました")

52341レース分の結果ファイルのリストを作成しました


In [10]:
import re
import pandas as pd
import glob

In [15]:
def makehorseIDList(files):
    """
    filesに指定されたレース結果からユニークなhorseIDのリストを作成する
    """
    horseIDList = []

    cnt = 1
    for file in files:
        print("\r" + f"horseIDListの作成中({cnt}/{len(files)})",end="")
        filename = str(file.split("/")[-1:])
        raceID = re.sub(r"\D+", "", filename)

        horseIDTmp = pd.read_pickle(f"../data/rawData/raceResults/{raceID}.pickle")
        horseIDTmp2 = horseIDTmp["horseID"].unique()
        horseIDList.extend(horseIDTmp2)
        cnt+=1
    
    horseIDListDF = pd.DataFrame(horseIDList)
    horseIDListFin = horseIDListDF[0].unique().tolist()

    print("\n" + f"{len(horseIDListFin)}頭分のhorseIDListを作成しました")
    return horseIDListFin

In [16]:
horseIDList = makehorseIDList(files)

horseIDListの作成中(52341/52341)
78112頭分のhorseIDListを作成しました


In [21]:
from bs4 import BeautifulSoup

In [39]:
horseID = "2006106019"

In [42]:
with open(f"../data/html/horse/{horseID}.txt","r",encoding="shift-jis") as f: #ファイルを開く
    html = f.read()  

In [43]:
html

'<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml" lang="ja" xml:lang="ja" id="html">\n<head>\n\n<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">\n<meta http-equiv="content-type" content="text/html; charset=euc-jp" />\n<meta http-equiv="content-script-type" content="text/javascript" />\n<meta http-equiv="content-style-type" content="text/css" />\n<link href="https://cdn.netkeiba.com/img.db/common/css/reset.css?20160421" rel="stylesheet" type="text/css" media="all" />\n<link href="https://cdn.netkeiba.com/img.db/common/css/common.css?20210819" rel="stylesheet" type="text/css" media="all" />\n<link href="https://cdn.netkeiba.com/img.db/common/css/db_detail.css?20180621" rel="stylesheet" type="text/css" media="all" />\n<link href="https://cdn.netkeiba.com/img.db/common/css/horse_detail.css?20250620" rel="stylesheet" type="text/css" media="all" />\n<link href="h

In [25]:
with open(f"../data/html/ped/{horseID}.txt","r",encoding="utf-8") as f: #ファイルを開く
    html = f.read()  

In [44]:
soup = BeautifulSoup(html,"html.parser")

In [45]:
soup.text

"\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nビービーボルト (B B Bolt) | 競走馬データ - netkeiba\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nお客様のブラウザはジャバスクリプト（JavaScript）に対応していないか無効になっています。\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nプレミアムサービス\n\n\n\n\nお気に入り馬\n\n\n\n\nメモ\n\n\n\n\nログイン/会員登録\n\n\n\n\n(s)ログイン\n\n\n\n\n(s)無料会員登録\n\n\n\n(s)ログアウト\n\n\n\n\n\nLIVE\n競輪\n\n\n\n\n\n\n\n\nトップ\n\n\nニュース\n\n\nレース\n\n\nA I\n\n\n\n予想\n\n\nUMAIビルダー\n\n\nコラム\n\n\nnetkeibaTV\n\n\n地方競馬\n\n\nデータベース\n\n\nショップ\n\n\n競馬新聞\n\n\n俺プロ\n\n\n一口馬主\n\n\nPOG\n\n\nまとめ\n\n\n\n\n\n\n\n\n\n\n\n競馬データTOP\n競走馬\n騎手\n調教師\n馬主\n生産者\nレース\nクラシック\n勝ち馬サーチ\n\n\n\n\n\n\n\n\n\n\n\n\nコミュニティ健全化に向けた取り組みについて\n\n\nいま競輪が熱い！ netkeirinで競輪を気軽に楽しもう\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nビービーボルト\n\nB B Bolt\n\n抹消\u3000セ\u3000鹿毛 \n\n\n共有\n\n\n\n\n\n\nお気に入り馬登録\n\n\n\n0人\n\n\n\nシェアする\n\n\n\n\n\nX\n\n\n\n\n\nFacebook\n\n\n\n\n\nLINE\n\n\n

In [46]:
tgtfilelist = glob.glob("../data/html/horse/*")

In [52]:
tgtfilelist = glob.glob("../data/html/ped/*")

In [47]:
len(tgtfilelist)

78335

In [50]:
from tqdm.notebook import tqdm

In [53]:
for file in tqdm(tgtfilelist):
    try:
        with open(file,"r",encoding="shift-jis") as f:
            txt = f.read()

        with  open(file,"w",encoding="utf-8") as f:
            f.write(txt)
    except Exception as e:
        pass
        

  0%|          | 0/78335 [00:00<?, ?it/s]

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd
import glob